# iFogSim — analiza eksperymentów

Notebook generuje wykresy PNG do `tex/img/` oraz tabele podsumowujące do `analysis/summary_tables.json` na podstawie plików CSV w `results/`.

Uruchomienie headless:
```bash
.venv/bin/jupyter nbconvert --execute --to notebook analysis/experiment_analysis.ipynb
```

## Ścieżki i zależności

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid", palette="colorblind")
except ImportError:
    sns = None

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "results"
IMG_DIR = PROJECT_ROOT / "tex" / "img"

LOOP2_ARTIFACT = 3.02
LOOP2_ARTIFACT_TOL = 0.01
CLOUD_COLOR = "#4C72B0"
FOG_COLOR = "#DD8452"

# DCNS: friendly labels (CSV columns stay Loop1Latency / Loop2Latency)
DCNS_TRACKING_LABEL = "Detection & tracking"
DCNS_PTZ_LABEL = "PTZ control"

IMG_DIR.mkdir(parents=True, exist_ok=True)

## Helpers — wspólne funkcje wykresów

In [2]:
def pct_reduction(fog_val, cloud_val):
    if cloud_val == 0:
        return np.nan
    return (cloud_val - fog_val) / cloud_val * 100


def strategy_rows(
    df: pd.DataFrame, filters: dict[str, object] | None = None
) -> tuple[pd.Series | None, pd.Series | None]:
    subset = df
    if filters:
        for column, value in filters.items():
            subset = subset[subset[column] == value]
    cloud = subset[subset["Strategy"] == "Cloud"]
    fog = subset[subset["Strategy"] == "Fog"]
    cloud_row = None if cloud.empty else cloud.iloc[0]
    fog_row = None if fog.empty else fog.iloc[0]
    return cloud_row, fog_row


def metric_reduction(cloud_row: pd.Series, fog_row: pd.Series, column: str) -> float:
    return pct_reduction(fog_row[column], cloud_row[column])


def mean_metric_reductions_by_group(
    df: pd.DataFrame,
    group_col: str,
    latency_col: str,
    energy_col: str,
) -> tuple[float, float, int]:
    latency_reductions: list[float] = []
    energy_reductions: list[float] = []
    for group_value in df[group_col].unique():
        cloud_row, fog_row = strategy_rows(df, {group_col: group_value})
        if cloud_row is None or fog_row is None:
            continue
        latency_reductions.append(metric_reduction(cloud_row, fog_row, latency_col))
        energy_reductions.append(metric_reduction(cloud_row, fog_row, energy_col))
    return np.nanmean(latency_reductions), np.nanmean(energy_reductions), len(latency_reductions)


def save_figure(fig: plt.Figure, path: Path, dpi: int = 150) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    plt.close(fig)


def plot_cloud_fog_lines(
    df: pd.DataFrame,
    x: str,
    y: str,
    title: str,
    xlabel: str,
    ylabel: str,
    out: Path,
    legend_loc: str = "best",
    ylim_bottom: float | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for strategy, marker, color in [
        ("Cloud", "o", CLOUD_COLOR),
        ("Fog", "s", FOG_COLOR),
    ]:
        sub = df[df["Strategy"] == strategy].sort_values(x)
        ax.plot(
            sub[x],
            sub[y],
            marker=marker,
            label=strategy,
            color=color,
            linewidth=1.5,
            markersize=6,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc=legend_loc)
    ax.grid(True, alpha=0.3)
    if ylim_bottom is not None:
        ax.set_ylim(bottom=ylim_bottom)
    save_figure(fig, out)


def plot_cloud_fog_bars_by_config(
    df: pd.DataFrame,
    configs: list[str],
    metrics: list[tuple[str, str]],
    suptitle: str,
    out: Path,
    symlog_cols: set[str] | None = None,
) -> None:
    symlog_cols = symlog_cols or set()
    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 5))
    x = np.arange(len(configs))
    width = 0.35
    for ax, (col, title) in zip(axes, metrics):
        cloud_vals = []
        fog_vals = []
        for cfg in configs:
            cloud_row, fog_row = strategy_rows(df, {"Config": cfg})
            cloud_vals.append(cloud_row[col] if cloud_row is not None else 0)
            fog_vals.append(fog_row[col] if fog_row is not None else 0)
        ax.bar(x - width / 2, cloud_vals, width, label="Cloud", color=CLOUD_COLOR)
        ax.bar(x + width / 2, fog_vals, width, label="Fog", color=FOG_COLOR)
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=7)
        if col in symlog_cols:
            ax.set_yscale("symlog", linthresh=1)
    axes[0].legend()
    fig.suptitle(suptitle)
    save_figure(fig, out)

## Ładowanie danych

In [3]:
class ScenarioConfig:
    def __init__(
        self,
        name: str,
        csv_name: str,
        expected_rows: int,
        annotate_fn: Callable[[pd.DataFrame], pd.DataFrame],
    ):
        self.name = name
        self.csv_name = csv_name
        self.expected_rows = expected_rows
        self.annotate_fn = annotate_fn


SCENARIOS: dict[str, ScenarioConfig] = {
    "DCNS": ScenarioConfig("DCNS", "results_DCNS.csv", 84, None),
    "CrowdSensing": ScenarioConfig("CrowdSensing", "results_CrowdSensing.csv", 56, None),
    "EnvironmentalMonitoring": ScenarioConfig(
        "EnvironmentalMonitoring",
        "results_EnvironmentalMonitoring.csv",
        56,
        None,
    ),
    "VRGame": ScenarioConfig("VRGame", "results_VRGame.csv", 28, None),
}


def load_datasets() -> dict[str, pd.DataFrame]:
    datasets: dict[str, pd.DataFrame] = {}
    for key, cfg in SCENARIOS.items():
        path = RESULTS_DIR / cfg.csv_name
        if not path.exists():
            print(f"Warning: {cfg.csv_name} not found, skipping {key}")
            continue
        df = pd.read_csv(path)
        if len(df) != cfg.expected_rows:
            print(
                f"Warning: {key}: expected {cfg.expected_rows} rows, got {len(df)} "
                "(full counts apply after env-sparse and vrgame experiments)"
            )
        datasets[key] = cfg.annotate_fn(df)
    return datasets

## DCNS — adnotacje i wykresy

Trzy mapy cieplne (% redukcji Fog vs Cloud): **detekcja i śledzenie**, sieć, energia.

**Co z tego wynika:**
- **Zielony (lewy górny róg):** Fog realnie wygrywa — niskie obciążenie, duża redukcja latencji i ruchu do chmury.
- **Żółty (środek):** Fog ≈ Cloud — degeneracja, brzeg nie daje korzyści mimo obecności infrastruktury.
- **x / ~ (prawy/dolny):** Fog się wysyca — brak ukończonej pętli detekcji lub mylący odczyt (~3 ms, nasycenie PTZ).
- **Panel energii:** prawie wszędzie blisko 0%; wyjątek 10x5 (−4%) — kompromis przy granicy MIPS routera.

In [4]:
def annotate_dcns_anomalies(dcns: pd.DataFrame) -> pd.DataFrame:
    df = dcns.copy()
    df["CamerasPerArea"] = df["NumCamerasPerArea"]
    df["TotalCameras"] = df["NumAreas"] * df["NumCamerasPerArea"]
    df["loop1_saturation_artifact"] = (
        (df["Loop1Latency(ms)"].sub(LOOP2_ARTIFACT).abs() <= LOOP2_ARTIFACT_TOL)
        & (df["CompletedLoops2"] == 0)
    )
    df["loop2_valid_ptz"] = (
        (df["Loop2Latency(ms)"].sub(LOOP2_ARTIFACT).abs() <= LOOP2_ARTIFACT_TOL)
        & (df["CompletedLoops2"] == 1)
    )
    df["loop1_plot"] = df["Loop1Latency(ms)"].where(~df["loop1_saturation_artifact"])
    df["fog_degeneracy"] = False
    fog_mask = df["Strategy"] == "Fog"
    fog = df.loc[fog_mask].copy()
    cloud = df[df["Strategy"] == "Cloud"].set_index(["NumAreas", "NumCamerasPerArea"])
    merged = fog.merge(
        cloud[["Loop1Latency(ms)", "NetworkUsage"]].rename(
            columns={
                "Loop1Latency(ms)": "cloud_loop1",
                "NetworkUsage": "cloud_net",
            }
        ),
        left_on=["NumAreas", "NumCamerasPerArea"],
        right_index=True,
        how="left",
    )
    degeneracy = (
        (merged["Loop1Latency(ms)"].sub(merged["cloud_loop1"]).abs() < 0.01)
        & (merged["NetworkUsage"].sub(merged["cloud_net"]).abs() < 1)
    )
    df.loc[fog_mask, "fog_degeneracy"] = degeneracy.values
    df["critical_failure"] = (
        (df["Strategy"] == "Fog")
        & (df["CompletedLoops1"] == 0)
        & (df["CompletedLoops2"] == 0)
    )
    return df


SCENARIOS["DCNS"].annotate_fn = annotate_dcns_anomalies


def _dcns_indexed_frames(dcns: pd.DataFrame):
    cloud = dcns[dcns["Strategy"] == "Cloud"].set_index(["NumAreas", "NumCamerasPerArea"])
    fog = dcns[dcns["Strategy"] == "Fog"].set_index(["NumAreas", "NumCamerasPerArea"])
    configs = sorted(cloud.index.intersection(fog.index))
    areas = sorted({area for area, _ in configs})
    cameras = sorted({cameras_per_area for _, cameras_per_area in configs})
    return cloud, fog, configs, areas, cameras


def _dcns_cell_value(
    fog_row: pd.Series, cloud_row: pd.Series, column: str, panel_kind: str
) -> tuple[float, str]:
    if panel_kind == "loop1":
        if fog_row["critical_failure"] or fog_row["Loop1Latency(ms)"] < 0:
            return np.nan, "fail"
        if fog_row["loop1_saturation_artifact"]:
            return np.nan, "artifact"
    return pct_reduction(fog_row[column], cloud_row[column]), ""


def _dcns_benefit_matrix(
    fog: pd.DataFrame,
    cloud: pd.DataFrame,
    configs: list[tuple[int, int]],
    areas: list[int],
    cameras: list[int],
    column: str,
    panel_kind: str,
) -> tuple[np.ndarray, np.ndarray]:
    mat = np.full((len(areas), len(cameras)), np.nan)
    overlay = np.full((len(areas), len(cameras)), "", dtype=object)
    for area_index, area in enumerate(areas):
        for camera_index, cameras_per_area in enumerate(cameras):
            key = (area, cameras_per_area)
            if key not in configs:
                continue
            fog_row = fog.loc[key]
            cloud_row = cloud.loc[key]
            mat[area_index, camera_index], overlay[area_index, camera_index] = _dcns_cell_value(
                fog_row, cloud_row, column, panel_kind
            )
    return mat, overlay


def _dcns_draw_panel_overlays(
    ax,
    mat: np.ndarray,
    overlay: np.ndarray,
    areas: list[int],
    cameras: list[int],
    anchors: set[tuple[int, int]],
) -> None:
    from matplotlib.patches import Rectangle

    for area_index, area in enumerate(areas):
        for camera_index, cameras_per_area in enumerate(cameras):
            mark = overlay[area_index, camera_index]
            if mark == "fail":
                ax.text(
                    camera_index,
                    area_index,
                    "x",
                    ha="center",
                    va="center",
                    color="#C44E52",
                    fontsize=13,
                    fontweight="bold",
                )
            elif mark == "artifact":
                ax.text(
                    camera_index,
                    area_index,
                    "~",
                    ha="center",
                    va="center",
                    color="#555555",
                    fontsize=14,
                )
            elif (area, cameras_per_area) in anchors and mark == "":
                ax.text(
                    camera_index,
                    area_index,
                    f"{mat[area_index, camera_index]:.0f}",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color="black",
                )

    for area, cameras_per_area in anchors:
        if area in areas and cameras_per_area in cameras:
            area_index = areas.index(area)
            camera_index = cameras.index(cameras_per_area)
            ax.add_patch(
                Rectangle(
                    (camera_index - 0.5, area_index - 0.5),
                    1,
                    1,
                    fill=False,
                    edgecolor="black",
                    linewidth=2,
                )
            )


def chart_dcns_loop1(dcns: pd.DataFrame, out: Path) -> None:
    """Three heatmaps: Fog vs Cloud improvement for latency, network, and energy."""
    from matplotlib.colors import TwoSlopeNorm
    from matplotlib.patches import Patch, Rectangle

    cloud, fog, configs, areas, cameras = _dcns_indexed_frames(dcns)
    anchors = {(10, 3), (10, 5), (10, 10)}
    panels: list[tuple[str, str, str]] = [
        (DCNS_TRACKING_LABEL, "Loop1Latency(ms)", "loop1"),
        ("Network usage", "NetworkUsage", "network"),
        ("Total energy", "TotalEnergy(W)", "energy"),
    ]
    norm = TwoSlopeNorm(vmin=-15, vcenter=0, vmax=100)

    fig = plt.figure(figsize=(14, 8))
    fig.text(
        0.5,
        0.975,
        "DCNS: Fog vs Cloud benefit map (all tested configurations)",
        ha="center",
        va="top",
        fontsize=13,
        fontweight="bold",
    )
    gs = fig.add_gridspec(
        nrows=3,
        ncols=3,
        height_ratios=[1.0, 0.055, 0.065],
        hspace=0.50,
        wspace=0.32,
        left=0.07,
        right=0.98,
        top=0.86,
        bottom=0.08,
    )
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    cbar_ax = fig.add_subplot(gs[1, :])
    legend_ax = fig.add_subplot(gs[2, :])
    legend_ax.axis("off")

    last_im = None
    for ax, (title, column, panel_kind) in zip(axes, panels):
        mat, overlay = _dcns_benefit_matrix(
            fog, cloud, configs, areas, cameras, column, panel_kind
        )
        last_im = ax.imshow(
            np.ma.masked_invalid(mat),
            aspect="auto",
            cmap="RdYlGn",
            norm=norm,
            origin="upper",
        )
        _dcns_draw_panel_overlays(ax, mat, overlay, areas, cameras, anchors)
        ax.set_xticks(range(len(cameras)))
        ax.set_xticklabels(cameras, fontsize=7)
        ax.set_xlabel("Cameras per area", fontsize=9)
        ax.set_title(title, fontsize=10, pad=6)

    axes[0].set_yticks(range(len(areas)))
    axes[0].set_yticklabels(areas)
    axes[0].set_ylabel("Number of areas")
    for ax in axes[1:]:
        ax.sharey(axes[0])
        plt.setp(ax.get_yticklabels(), visible=False)

    cbar = fig.colorbar(last_im, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("% reduction, Fog vs Cloud (green = Fog better)", labelpad=4)
    cbar.ax.tick_params(labelsize=8)

    legend_ax.legend(
        handles=[
            Patch(facecolor="none", edgecolor="black", linewidth=2, label="Report anchors"),
            Patch(facecolor="none", label="x tracking loop failed"),
            Patch(facecolor="none", label="~ invalid tracking reading (PTZ saturation)"),
        ],
        loc="center",
        ncol=3,
        frameon=False,
        fontsize=9,
    )
    fig.savefig(out, dpi=150)
    plt.close(fig)


def chart_dcns_energy_anomaly(dcns: pd.DataFrame, out: Path) -> None:
    cloud_row, fog_row = strategy_rows(
        dcns, {"NumAreas": 10, "NumCamerasPerArea": 5}
    )
    if cloud_row is None or fog_row is None:
        return

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metrics = [
        ("Loop1Latency(ms)", f"{DCNS_TRACKING_LABEL} (ms)"),
        ("NetworkUsage", "Network usage"),
        ("TotalEnergy(W)", "Total energy (W)"),
    ]
    for ax, (column, title) in zip(axes, metrics):
        cloud_val = cloud_row[column]
        fog_val = fog_row[column]
        ax.bar(["Cloud", "Fog"], [cloud_val, fog_val], color=[CLOUD_COLOR, FOG_COLOR])
        ax.set_title(title)
        if column == "TotalEnergy(W)":
            pct = (fog_val - cloud_val) / cloud_val * 100
            ax.annotate(f"+{pct:.1f}% vs Cloud", xy=(1, fog_val), ha="center", va="bottom", fontsize=9)
        elif column == "Loop1Latency(ms)":
            reduction = pct_reduction(fog_val, cloud_val)
            ax.annotate(f"−{reduction:.1f}% vs Cloud", xy=(1, fog_val), ha="center", va="bottom", fontsize=9)
    fig.suptitle("DCNS 10x5: Cloud vs Fog (latency, network, energy)")
    save_figure(fig, out)

## CrowdSensing — adnotacje i wykresy

In [5]:
def annotate_crowd(crowd: pd.DataFrame) -> pd.DataFrame:
    df = crowd.copy()
    df["latency_valid"] = df["AvgLoopLatency(ms)"] >= 0
    df["Users"] = df["NumMobileUsers"]
    return df


SCENARIOS["CrowdSensing"].annotate_fn = annotate_crowd


def chart_crowd_network_energy(crowd: pd.DataFrame, out: Path) -> None:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    for strategy, color in [("Cloud", CLOUD_COLOR), ("Fog", FOG_COLOR)]:
        sub = crowd[crowd["Strategy"] == strategy].sort_values("Users")
        ax1.plot(sub["Users"], sub["NetworkUsage"], marker="o", label=strategy, color=color)
        ax2.plot(
            sub["Users"],
            sub["TotalEnergy(W)"] / 1e6,
            marker="o",
            label=strategy,
            color=color,
        )
    ax1.set_ylabel("Network usage")
    ax1.set_title("CrowdSensing: network usage and energy vs scale")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.set_xlabel("Number of mobile users")
    ax2.set_ylabel("Total energy (x10⁶ W)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    save_figure(fig, out)

## EnvironmentalMonitoring — adnotacje i wykresy

In [6]:
def annotate_env(env: pd.DataFrame) -> pd.DataFrame:
    df = env.copy()
    if "AnomalySelectivity" not in df.columns:
        df["AnomalySelectivity"] = 1.0
    df["TotalSensors"] = df["NumGateways"] * df["SensorsPerGateway"]
    df["Config"] = df["NumGateways"].astype(str) + "x" + df["SensorsPerGateway"].astype(str)
    return df


SCENARIOS["EnvironmentalMonitoring"].annotate_fn = annotate_env


def _env_baseline(env: pd.DataFrame) -> pd.DataFrame:
    if "AnomalySelectivity" not in env.columns:
        return env
    baseline = env[env["AnomalySelectivity"] == 1.0]
    return baseline if len(baseline) else env


def chart_env_fog_vs_cloud(env: pd.DataFrame, out: Path) -> None:
    sub = _env_baseline(env)
    configs = list(sub["Config"].unique())
    plot_cloud_fog_bars_by_config(
        sub,
        configs,
        [
            ("AvgLoopLatency(ms)", "Avg loop latency (ms)"),
            ("NetworkUsage", "Network usage"),
            ("TotalEnergy(W)", "Total energy (W)"),
        ],
        "EnvironmentalMonitoringFog: Cloud vs Fog (14 configurations)",
        out,
        symlog_cols={"NetworkUsage"},
    )


def chart_env_selectivity_comparison(env: pd.DataFrame, out: Path) -> None:
    if "AnomalySelectivity" not in env.columns:
        print("Warning: env_selectivity_comparison skipped (no AnomalySelectivity column)")
        return
    selectivities = sorted(env["AnomalySelectivity"].unique())
    if not any(abs(s - 0.1) < 0.01 for s in selectivities):
        print("Warning: env_selectivity_comparison skipped (need selectivity 0.1 data)")
        return

    summary_rows = []
    for selectivity in [1.0, 0.1]:
        subset = env[env["AnomalySelectivity"].sub(selectivity).abs() < 0.01]
        if subset.empty:
            continue
        avg_latency, avg_energy, config_count = mean_metric_reductions_by_group(
            subset,
            "Config",
            "AvgLoopLatency(ms)",
            "TotalEnergy(W)",
        )
        if config_count == 0:
            continue
        summary_rows.append(
            {
                "label": f"selectivity={selectivity:g}",
                "latency": avg_latency,
                "energy": avg_energy,
            }
        )

    if len(summary_rows) < 2:
        print("Warning: env_selectivity_comparison skipped (incomplete selectivity pairs)")
        return

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    labels = [row["label"] for row in summary_rows]
    colors = [CLOUD_COLOR, FOG_COLOR]
    latency_vals = [row["latency"] for row in summary_rows]
    energy_vals = [row["energy"] for row in summary_rows]
    bars0 = axes[0].bar(labels, latency_vals, color=colors[: len(labels)])
    axes[0].set_title("Avg latency reduction (Fog vs Cloud)")
    axes[0].set_ylabel("Reduction (%)")
    axes[0].bar_label(bars0, labels=[f"{v:.2f}%" for v in latency_vals], padding=3, fontsize=9)
    bars1 = axes[1].bar(labels, energy_vals, color=colors[: len(labels)])
    axes[1].set_title("Avg energy reduction (Fog vs Cloud)")
    axes[1].set_ylabel("Reduction (%)")
    axes[1].bar_label(bars1, labels=[f"{v:.2f}%" for v in energy_vals], padding=3, fontsize=9)
    for ax, vals in [(axes[0], latency_vals), (axes[1], energy_vals)]:
        ymax = max(vals) if vals else 1
        ax.set_ylim(0, ymax * 1.12)
    fig.suptitle("EnvironmentalMonitoring: anomaly selectivity 1.0 vs 0.1")
    save_figure(fig, out)

## VRGame — adnotacje i wykresy

In [7]:
def annotate_vrgame(vrgame: pd.DataFrame) -> pd.DataFrame:
    df = vrgame.copy()
    df["Scale"] = df["NumDepartments"] * df["MobilesPerDept"]
    df["Config"] = df["NumDepartments"].astype(str) + "x" + df["MobilesPerDept"].astype(str)
    return df


SCENARIOS["VRGame"].annotate_fn = annotate_vrgame


VRGAME_KEY_CONFIGS = ["1x1", "12x4", "10x5"]


def chart_vrgame_key_configs(vrgame: pd.DataFrame, out: Path) -> None:
    plot_cloud_fog_bars_by_config(
        vrgame,
        VRGAME_KEY_CONFIGS,
        [
            ("AvgLoopLatency(ms)", "Avg loop latency (ms)"),
            ("NetworkUsage", "Network usage"),
        ],
        "VRGame: Cloud vs Fog at key configurations",
        out,
        symlog_cols={"NetworkUsage"},
    )

## Ładowanie danych i podgląd

In [8]:
from IPython.display import display

datasets = load_datasets()
print("Row counts:", {k: len(v) for k, v in datasets.items()})
for name, df in datasets.items():
    print(name)
    display(df.head(2))

Row counts: {'DCNS': 84, 'CrowdSensing': 56, 'EnvironmentalMonitoring': 56, 'VRGame': 28}
DCNS


,AppType,Strategy,NumAreas,NumCamerasPerArea,Loop1Latency(ms),Loop2Latency(ms),CompletedLoops1,CompletedLoops2,NetworkUsage,TotalEnergy(W),CloudExecutionCost,CamerasPerArea,TotalCameras,loop1_saturation_artifact,loop2_valid_ptz,loop1_plot,fog_degeneracy,critical_failure
0,DCNS,Cloud,1,1,109.5930,105.04,1,1,40452.6,3.751334e+06,828487.3200,1,1,False,False,109.5930,False,False
1,DCNS,Fog,1,1,2.9343,3.02,1,1,2411.9,3.180971e+06,3741.6857,1,1,False,True,2.9343,False,False


CrowdSensing


,AppType,Strategy,NumMobileUsers,AvgLoopLatency(ms),NetworkUsage,TotalEnergy(W),MigrationTime(ms),CloudExecutionCost,latency_valid,Users
0,CrowdSensing,Cloud,1,-1.0,5141.000,2.453404e+07,0.00,17721.6150,False,1
1,CrowdSensing,Fog,1,-1.0,9510.704,2.631652e+07,7.04,10854.7479,False,1


EnvironmentalMonitoring


,AppType,Strategy,NumGateways,SensorsPerGateway,AnomalySelectivity,AvgLoopLatency(ms),NetworkUsage,TotalEnergy(W),CloudExecutionCost,TotalSensors,Config
0,EnvironmentalMonitoring,Cloud,1,1,1.0,215.0723,5413.15,3.580922e+06,824946.9771,1,1x1
1,EnvironmentalMonitoring,Fog,1,1,1.0,1.1986,0.00,3.000351e+06,0.0000,1,1x1


VRGame


,AppType,Strategy,NumDepartments,MobilesPerDept,AvgLoopLatency(ms),NetworkUsage,TotalEnergy(W),CloudExecutionCost,Scale,Config
0,VRGame,Cloud,1,1,228.0172,19077.5,3.185252e+06,18687.04,1,1x1
1,VRGame,Fog,1,1,20.3483,2332.5,3.188720e+06,6066.08,1,1x1


### DCNS — flagi anomalii

In [9]:
if "DCNS" in datasets:
    dcns = datasets["DCNS"]
    print("DCNS saturation artifacts:", int(dcns["loop1_saturation_artifact"].sum()))
    print("Critical failures:", int(dcns["critical_failure"].sum()))
    print("Fog degeneracy:", int(dcns["fog_degeneracy"].sum()))

DCNS saturation artifacts: 10
Critical failures: 1
Fog degeneracy: 16


### DCNS — benefit map (detection & tracking / network / energy)

Zastępuje wykres liniowy: cała przestrzeń konfiguracji w jednym obrazie.

In [10]:
created: list[str] = []

if "DCNS" in datasets:
    out = IMG_DIR / "dcns_loop1_vs_cameras.png"
    chart_dcns_loop1(datasets["DCNS"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/dcns_loop1_vs_cameras.png


### DCNS 10x5 — anomalia energii

In [11]:
if "DCNS" in datasets:
    out = IMG_DIR / "dcns_energy_anomaly.png"
    chart_dcns_energy_anomaly(datasets["DCNS"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/dcns_energy_anomaly.png


### CrowdSensing — sieć i energia

In [12]:
if "CrowdSensing" in datasets:
    out = IMG_DIR / "crowd_network_energy.png"
    chart_crowd_network_energy(datasets["CrowdSensing"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/crowd_network_energy.png


### EnvironmentalMonitoring — Fog vs Cloud

In [13]:
if "EnvironmentalMonitoring" in datasets:
    out = IMG_DIR / "environmental_monitoring_fog_vs_cloud.png"
    chart_env_fog_vs_cloud(datasets["EnvironmentalMonitoring"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/environmental_monitoring_fog_vs_cloud.png


### EnvironmentalMonitoring — selectivity 1.0 vs 0.1

In [14]:
if "EnvironmentalMonitoring" in datasets:
    out = IMG_DIR / "env_selectivity_comparison.png"
    chart_env_selectivity_comparison(datasets["EnvironmentalMonitoring"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/env_selectivity_comparison.png


### VRGame — kluczowe konfiguracje

In [15]:
if "VRGame" in datasets:
    out = IMG_DIR / "vrgame_key_configs.png"
    chart_vrgame_key_configs(datasets["VRGame"], out)
    if out.exists():
        created.append(str(out.relative_to(PROJECT_ROOT)))
        print(f"Saved {out.relative_to(PROJECT_ROOT)}")

Saved tex/img/vrgame_key_configs.png


## Tabele podsumowujące

In [16]:
def _summary_dcns_regimes(dcns: pd.DataFrame) -> pd.DataFrame:
    anchors = [
        (10, 3, "Fog dominance"),
        (10, 5, "Partial efficiency / energy anomaly"),
        (10, 10, "Critical failure"),
        (2, 25, "Fog degeneracy"),
    ]
    rows = []
    for areas, cameras_per_area, regime in anchors:
        cloud_row, fog_row = strategy_rows(
            dcns, {"NumAreas": areas, "NumCamerasPerArea": cameras_per_area}
        )
        for strategy, row in [("Cloud", cloud_row), ("Fog", fog_row)]:
            if row is None:
                continue
            rows.append(
                {
                    "Regime": regime,
                    "Config": f"{areas}x{cameras_per_area}",
                    "Strategy": strategy,
                    "Tracking_ms": row["Loop1Latency(ms)"],
                    "PTZ_ms": row["Loop2Latency(ms)"],
                    "Network": row["NetworkUsage"],
                    "Energy_W": row["TotalEnergy(W)"],
                }
            )
    return pd.DataFrame(rows)


def _summary_crowd_crossover(crowd: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for users in [18, 20, 25, 50, 52]:
        cloud_row, fog_row = strategy_rows(crowd, {"Users": users})
        if cloud_row is None or fog_row is None:
            continue
        rows.append(
            {
                "Users": users,
                "Cloud_Net": cloud_row["NetworkUsage"],
                "Fog_Net": fog_row["NetworkUsage"],
                "Fog_vs_Cloud_pct": metric_reduction(
                    cloud_row, fog_row, "NetworkUsage"
                ),
            }
        )
    return pd.DataFrame(rows)


def _summary_env_fog_benefit(env: pd.DataFrame) -> pd.DataFrame:
    baseline = _env_baseline(env)
    rows = []
    for cfg in baseline["Config"].unique():
        cloud_row, fog_row = strategy_rows(baseline, {"Config": cfg})
        if cloud_row is None or fog_row is None:
            continue
        rows.append(
            {
                "Config": cfg,
                "Latency_reduction_pct": metric_reduction(
                    cloud_row, fog_row, "AvgLoopLatency(ms)"
                ),
                "Network_reduction_pct": metric_reduction(
                    cloud_row, fog_row, "NetworkUsage"
                ),
                "Energy_reduction_pct": metric_reduction(
                    cloud_row, fog_row, "TotalEnergy(W)"
                ),
                "Fog_Network": fog_row["NetworkUsage"],
            }
        )
    return pd.DataFrame(rows)


def _summary_env_selectivity(env: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for selectivity in sorted(env["AnomalySelectivity"].unique()):
        subset = env[env["AnomalySelectivity"].sub(selectivity).abs() < 0.01]
        avg_latency, avg_energy, config_count = mean_metric_reductions_by_group(
            subset,
            "Config",
            "AvgLoopLatency(ms)",
            "TotalEnergy(W)",
        )
        rows.append(
            {
                "AnomalySelectivity": selectivity,
                "Avg_latency_reduction_pct": avg_latency,
                "Avg_energy_reduction_pct": avg_energy,
                "Config_count": config_count,
            }
        )
    return pd.DataFrame(rows)


def _summary_vrgame_scale(vrgame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for scale in sorted(vrgame["Scale"].unique()):
        cloud_row, fog_row = strategy_rows(vrgame, {"Scale": scale})
        if cloud_row is None or fog_row is None:
            continue
        rows.append(
            {
                "Scale": scale,
                "NumDepartments": int(cloud_row["NumDepartments"]),
                "MobilesPerDept": int(cloud_row["MobilesPerDept"]),
                "Cloud_latency_ms": cloud_row["AvgLoopLatency(ms)"],
                "Fog_latency_ms": fog_row["AvgLoopLatency(ms)"],
                "Latency_reduction_pct": metric_reduction(
                    cloud_row, fog_row, "AvgLoopLatency(ms)"
                ),
                "Energy_reduction_pct": metric_reduction(
                    cloud_row, fog_row, "TotalEnergy(W)"
                ),
            }
        )
    return pd.DataFrame(rows)


def build_summary_tables(datasets: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    if "DCNS" in datasets:
        tables["dcns_regimes"] = _summary_dcns_regimes(datasets["DCNS"])
    if "CrowdSensing" in datasets:
        tables["crowd_crossover"] = _summary_crowd_crossover(datasets["CrowdSensing"])
    if "EnvironmentalMonitoring" in datasets:
        env = datasets["EnvironmentalMonitoring"]
        tables["env_fog_benefit"] = _summary_env_fog_benefit(env)
        if "AnomalySelectivity" in env.columns:
            tables["env_selectivity"] = _summary_env_selectivity(env)
    if "VRGame" in datasets:
        tables["vrgame_scale"] = _summary_vrgame_scale(datasets["VRGame"])
    return tables


tables = build_summary_tables(datasets)
for name, table in tables.items():
    print(f"\n=== {name} ===")
    display(table)


=== dcns_regimes ===


,Regime,Config,Strategy,Tracking_ms,PTZ_ms,Network,Energy_W
0,Fog dominance,10x3,Cloud,260.3094,105.04,1015587.0,1.014080e+07
1,Fog dominance,10x3,Fog,7.2200,3.02,86637.0,1.007455e+07
2,Partial efficiency / energy anomaly,10x5,Cloud,566.5074,105.04,1046703.0,1.352380e+07
3,Partial efficiency / energy anomaly,10x5,Fog,109.5648,105.04,234688.0,1.401849e+07
4,Critical failure,10x10,Cloud,796.1587,105.04,1124493.0,2.198954e+07
5,Critical failure,10x10,Fog,-1.0000,-1.00,1116784.0,2.147923e+07
6,Fog degeneracy,2x25,Cloud,566.5058,105.04,1046684.6,1.218486e+07
7,Fog degeneracy,2x25,Fog,566.5104,105.04,1046684.6,1.219637e+07



=== crowd_crossover ===


,Users,Cloud_Net,Fog_Net,Fog_vs_Cloud_pct
0,18,92538.000,93909.646,-1.482252
1,20,102820.000,102253.112,0.551340
2,25,128525.000,119797.726,6.790332
3,50,256589.580,240860.978,6.129868
4,52,265416.428,262145.158,1.232505



=== env_fog_benefit ===


,Config,Latency_reduction_pct,Network_reduction_pct,Energy_reduction_pct,Fog_Network
0,1x1,99.442699,100.0,16.212888,0.0
1,1x2,97.020490,100.0,15.953343,0.0
2,2x2,96.985600,100.0,15.040474,0.0
3,2x4,96.793382,100.0,15.040786,0.0
4,3x3,96.879511,100.0,14.224160,0.0
5,4x4,96.786873,100.0,13.489816,0.0
6,5x3,96.875124,100.0,12.825023,0.0
7,5x5,96.698956,100.0,12.825710,0.0
8,6x4,96.786780,100.0,12.221452,0.0
9,8x3,96.874788,100.0,11.164414,0.0



=== env_selectivity ===


,AnomalySelectivity,Avg_latency_reduction_pct,Avg_energy_reduction_pct,Config_count
0,0.1,96.983330,12.872272,14
1,1.0,97.009969,12.872272,14



=== vrgame_scale ===


,Scale,NumDepartments,MobilesPerDept,Cloud_latency_ms,Fog_latency_ms,Latency_reduction_pct,Energy_reduction_pct
0,1,1,1,228.0172,20.3483,91.075980,-0.108894
1,2,1,2,230.0532,22.1699,90.363142,10.969388
2,4,2,2,230.1182,22.1796,90.361649,10.831873
3,8,2,4,229.7030,34.0096,85.194098,9.331703
4,9,3,3,229.3368,22.6906,90.105993,7.900747
5,15,5,3,228.7885,22.8044,90.032541,4.885131
6,16,4,4,228.7810,35.9196,84.299570,5.539606
7,24,6,4,429.6995,34.3890,91.996965,3.250597
8,25,5,5,327.8812,328.5211,-0.195162,0.001915
9,40,8,5,788.8858,789.3170,-0.054659,0.008576


## Eksport JSON i kluczowe liczby raportu

In [17]:
def print_cloud_fog_comparison(
    label: str,
    cloud_row: pd.Series,
    fog_row: pd.Series,
    column: str,
    *,
    unit: str = "",
    value_fmt: str = ".2f",
) -> None:
    cloud_val = cloud_row[column]
    fog_val = fog_row[column]
    reduction = pct_reduction(fog_val, cloud_val)
    unit_suffix = f" {unit}" if unit else ""
    print(
        f"  {label}: Cloud {cloud_val:{value_fmt}}{unit_suffix} → "
        f"Fog {fog_val:{value_fmt}}{unit_suffix} ({reduction:.1f}% reduction)"
    )


def print_report_key_numbers(
    datasets: dict[str, pd.DataFrame],
    tables: dict[str, pd.DataFrame],
    created: list[str],
) -> None:
    print("\nPNG files created:")
    for path in created:
        print(f"  {path}")

    print("\nKey numbers for report:")
    if "DCNS" in datasets:
        dcns = datasets["DCNS"]
        cloud_row, fog_row = strategy_rows(
            dcns, {"NumAreas": 10, "NumCamerasPerArea": 3}
        )
        if cloud_row is not None and fog_row is not None:
            print_cloud_fog_comparison(
                f"DCNS 10x3 {DCNS_TRACKING_LABEL}",
                cloud_row,
                fog_row,
                "Loop1Latency(ms)",
                unit="ms",
            )
            print_cloud_fog_comparison(
                "DCNS 10x3 network",
                cloud_row,
                fog_row,
                "NetworkUsage",
                value_fmt=".0f",
            )

        cloud_row, fog_row = strategy_rows(
            dcns, {"NumAreas": 10, "NumCamerasPerArea": 5}
        )
        if cloud_row is not None and fog_row is not None:
            cloud_energy = cloud_row["TotalEnergy(W)"]
            fog_energy = fog_row["TotalEnergy(W)"]
            print(
                f"  DCNS 10x5 energy: Fog +"
                f"{(fog_energy - cloud_energy) / cloud_energy * 100:.1f}% vs Cloud (anomaly)"
            )

    if "crowd_crossover" in tables and not tables["crowd_crossover"].empty:
        crowd_crossover = tables["crowd_crossover"]
        for users, label in [(20, "Crowd CSV 20u"), (25, "Crowd CSV 25u: max benefit")]:
            row = crowd_crossover[crowd_crossover["Users"] == users]
            if not row.empty:
                print(f"  {label}: Fog_vs_Cloud {row.iloc[0]['Fog_vs_Cloud_pct']:.1f}%")

    if "EnvironmentalMonitoring" in datasets:
        env = datasets["EnvironmentalMonitoring"]
        cloud_row, fog_row = strategy_rows(
            env, {"NumGateways": 1, "SensorsPerGateway": 1}
        )
        if cloud_row is not None and fog_row is not None:
            print_cloud_fog_comparison(
                "EnvMonitoring 1x1 latency",
                cloud_row,
                fog_row,
                "AvgLoopLatency(ms)",
                unit="ms",
            )
        print("  EnvMonitoring Fog network=0 for all configs (local processing)")


summary_path = PROJECT_ROOT / "analysis" / "summary_tables.json"
summary_path.write_text(
    json.dumps({k: v.to_dict(orient="records") for k, v in tables.items()}, indent=2),
    encoding="utf-8",
)
print(f"Wrote {summary_path.relative_to(PROJECT_ROOT)}")
print_report_key_numbers(datasets, tables, created)

Wrote analysis/summary_tables.json

PNG files created:
  tex/img/dcns_loop1_vs_cameras.png
  tex/img/dcns_energy_anomaly.png
  tex/img/crowd_network_energy.png
  tex/img/environmental_monitoring_fog_vs_cloud.png
  tex/img/env_selectivity_comparison.png
  tex/img/vrgame_key_configs.png

Key numbers for report:
  DCNS 10x3 Detection & tracking: Cloud 260.31 ms → Fog 7.22 ms (97.2% reduction)
  DCNS 10x3 network: Cloud 1015587 → Fog 86637 (91.5% reduction)
  DCNS 10x5 energy: Fog +3.7% vs Cloud (anomaly)
  Crowd CSV 20u: Fog_vs_Cloud 0.6%
  Crowd CSV 25u: max benefit: Fog_vs_Cloud 6.8%
  EnvMonitoring 1x1 latency: Cloud 215.07 ms → Fog 1.20 ms (99.4% reduction)
  EnvMonitoring Fog network=0 for all configs (local processing)
